# 01 — Cleaned Data Validation

This notebook starts from the cleaned calibration-candidate dataset. It does not repeat raw-data cleaning. We verify the input schema, maturity coverage and numerical consistency before implementing the pricing model.


## 1. Project paths and imports

Import the libraries needed for data inspection and define a project-relative path. Keep the cleaned CSV in `data/cleaned/` so the notebook is portable across team members' machines.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "calibration_candidates.csv.gz"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned calibration dataset not found: {DATA_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")


## 2. Load the cleaned calibration candidates

Load the long-format calibration file. Each row represents one option quote; this is the appropriate structure for calibration across strikes and maturities.


In [ ]:
options = pd.read_csv(DATA_PATH)
print(f"Rows: {len(options):,} | Columns: {len(options.columns)}")
display(options.head())


## 3. Validate the required schema

Confirm that all fields required for European call calibration are present. Stop early with a readable error if a required field is missing.


In [ ]:
REQUIRED_COLUMNS = {
    "Strike", "Expiry", "Tau", "Spot", "Dividend_Yield",
    "Risk_Free_Rate_CC", "Market_Price", "Trading_Volume"
}
missing_columns = REQUIRED_COLUMNS.difference(options.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
print("Schema check passed.")
display(options[list(sorted(REQUIRED_COLUMNS))].dtypes)


## 4. Check maturity and quote coverage

Report observations by expiry and the number of distinct maturities. The cleaned sample currently contains 477 positive-volume candidates across five expiries; the code reports actual values rather than assuming them.


In [ ]:
maturity_summary = (
    options.groupby(["Expiry", "Days_To_Maturity"], dropna=False)
    .size().rename("Number_of_Quotes").reset_index()
    .sort_values("Days_To_Maturity")
)
print(f"Distinct expiries: {options['Expiry'].nunique()}")
display(maturity_summary)


## 5. Numerical consistency checks

Check that key inputs are finite and economically admissible. These checks complement, but do not replace, the original data-cleaning quality reports.


In [ ]:
numeric_columns = ["Strike", "Tau", "Spot", "Dividend_Yield", "Risk_Free_Rate_CC", "Market_Price"]
non_finite_counts = options[numeric_columns].apply(lambda column: (~np.isfinite(column)).sum())
checks = {
    "all_strikes_positive": bool((options["Strike"] > 0).all()),
    "all_maturities_positive": bool((options["Tau"] > 0).all()),
    "all_spot_values_positive": bool((options["Spot"] > 0).all()),
    "all_market_prices_non_negative": bool((options["Market_Price"] >= 0).all()),
    "all_trading_volumes_positive": bool((options["Trading_Volume"] > 0).all()),
    "all_numeric_inputs_finite": bool((non_finite_counts == 0).all()),
}
display(non_finite_counts.rename("Non_Finite_Count").to_frame())
display(pd.Series(checks, name="Passed"))
if not all(checks.values()):
    raise ValueError("One or more calibration input checks failed. Inspect the diagnostics above.")
print("All input validation checks passed.")


## 6. Calibration sample snapshot

Record the sample size, maturity range and market-input ranges. This is our checkpoint before deriving the risk-neutral characteristic function and implementing the option pricer in dedicated Python modules.


In [ ]:
sample_snapshot = pd.Series({
    "Number of calibration quotes": len(options),
    "Number of expiries": options["Expiry"].nunique(),
    "Minimum maturity (days)": options["Days_To_Maturity"].min(),
    "Maximum maturity (days)": options["Days_To_Maturity"].max(),
    "Minimum strike": options["Strike"].min(),
    "Maximum strike": options["Strike"].max(),
    "Minimum market price": options["Market_Price"].min(),
    "Maximum market price": options["Market_Price"].max(),
})
display(sample_snapshot.to_frame("Value"))
